# IFQ718 Assignment 2

---

## Questions

Select any three

1. What is the profit of each business per year? - First question

2. Which product generated the most profit for each business per year? - Second question

3. What was the most popular product for each business in each year? - Third question

4. Are there periods of the year where some businesses are more profitable?

5. What were the most utilised promotions for each business?

6. Which customers were most loyal for each business?

7. What is the total expenditure of loyal customers compared to one-off customers?

8. What is the employee turnover rate of each business?

9. How much has inflation impacted the profit margin of each business?

10. What is the impact of seasonal fluctuations on each business?

---

## Getting started

We have provided some code below to get you started

In [12]:
pip install tabulate

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import csv
import json
import glob
import pandas as pd
from datetime import datetime
import tabulate

---

# Preparations

If you wish, you may prepare frames or files prior to the Exploration sections.

For example, if you want to prepare a frame that is used in multiple Exploration sectons, you may prepare it here.

In [19]:
Receipts_Dir = 'receipts'

In [20]:
def format_df_for_display(df):
    df_display = df.copy()
    if 'Date' in df_display.columns:
        df_display['Date'] = df_display['Date'].dt.strftime('%y-%m-%d')

    return df_display

In [28]:
def load_and_flatten_receipts(directory_path):
    
    all_records = []
    
    file_paths = glob.glob(os.path.join(directory_path, '*.json'))
    for file_path in file_paths:
        try:
            with open(file_path, 'r') as f:
                full_data = json.load(f)

                receipt_data = full_data.get('root', full_data)

                business_name = receipt_data.get('Business', {}).get('Name', 'N/A')

                receipt_date_str = receipt_data.get('Date', 'N/A')

                receipt_total = receipt_data.get('Total', 'N/A')

                receipt_year = 'N/A'

                if receipt_date_str != 'N/A':
                    try:
                        date_obj = pd.to_datetime(receipt_date_str, errors = 'coerce')
                        if pd.notna(date_obj):
                            receipt_year = date_obj.year

                            receipt_date_str = date_obj.strftime('%Y/%m/%d')
                    except Exception:
                        pass

                products = receipt_data.get('Products', [])


                for item in products:
                    item_quantity = item.get('Quantity', 0)
                    item_unit_price = item.get('Price', 0.0)

                    record = {
                        'Receipt_ID': os.path.basename(file_path),
                        'Business_Name': business_name,
                        'Date': receipt_date_str,
                        'Year': receipt_year,
                        'Item_Name': item.get('Name', 'Unknown Item'),
                        'Item_Quantity': item_quantity,
                        'Item_Unit_Price': item_unit_price,
                        'Item_COGS': item.get('Cost', 0.0),
                        'Item_Sales_Revenue': item_unit_price * item_quantity,
                        'Receipt_Total': receipt_total
                    }
                    all_records.append(record)

        except Exception:
            pass

    df = pd.DataFrame(all_records)

    if 'Date' in df.columns and df['Date'].dtype == 'object' and df['Date'].iloc[0] != 'N/A':
        try:
            df['Date'] = pd.to_datetime(df['Date'], format = '%Y/%m/%d', errors = 'coerce')
        except: 
            pass

    return df

if not receipts_df.empty:
    print("\n Flattened Receipts Data Table (First 20 Rows for Reference)")
    
    df_for_display = format_df_for_display(receipts_df.head(20))
    
    print(df_for_display.to_markdown(index=False, numalign="left", stralign="left", floatfmt=".2f"))



 Flattened Receipts Data Table (First 20 Rows for Reference)
| Receipt_ID                | Business_Name        | Date     | Year   | Item_Name      | Item_Quantity   | Item_Unit_Price   | Item_COGS   | Item_Sales_Revenue   | Receipt_Total   | Item_Profit   |
|:--------------------------|:---------------------|:---------|:-------|:---------------|:----------------|:------------------|:------------|:---------------------|:----------------|:--------------|
| 2209696-260610-02056.json | Wake Up with Coffee  | 26-06-10 | 2026   | Ham toastie    | 2               | 7.90              | 5.00        | 15.80                | 45.54           | 10.80         |
| 2209696-260610-02056.json | Wake Up with Coffee  | 26-06-10 | 2026   | Long Black     | 3               | 4.40              | 2.90        | 13.20                | 45.54           | 10.30         |
| 2209696-260610-02056.json | Wake Up with Coffee  | 26-06-10 | 2026   | Cupcake        | 1               | 8.60              | 2.90        | 

---

## Exploration One

### Introduction

This first analysis provides a macro-level overview of financial health. It sums up all profits (Revenue - Cost of Goods Sold/COGS) across every item sold for each business in a given year. This allows for quick, year-over-year trending and comparison between different business entities.

### Methods

In [23]:
receipts_df = load_and_flatten_receipts(Receipts_Dir)

if not receipts_df.empty:
    receipts_df['Item_Profit'] = receipts_df['Item_Sales_Revenue'] - receipts_df['Item_COGS']

    profit_per_year = receipts_df.groupby(['Business_Name', 'Year'])['Item_Profit'].sum().reset_index()

    profit_per_year.rename(columns={'Item_Profit': 'Total_Yearly_Profit'}, inplace = True)

    print("\n Analysis for Focus Question 1: Profit per Business per Year")

    print(profit_per_year.to_markdown(index = False, floatfmt = ".2f", numalign = "left", stralign = "left"))


 Analysis for Focus Question 1: Profit per Business per Year
| Business_Name        | Year   | Total_Yearly_Profit   |
|:---------------------|:-------|:----------------------|
| Ed's Barber Supplies | 2022   | 12973.00              |
| Ed's Barber Supplies | 2023   | 32788.10              |
| Ed's Barber Supplies | 2024   | 39890.50              |
| Ed's Barber Supplies | 2025   | 11739.90              |
| Wake Up with Coffee  | 2022   | 6514.70               |
| Wake Up with Coffee  | 2023   | 11924.20              |
| Wake Up with Coffee  | 2024   | 17736.80              |
| Wake Up with Coffee  | 2025   | 16805.20              |
| Wake Up with Coffee  | 2026   | 14175.40              |
| Wake Up with Coffee  | 2027   | 20452.00              |
| Wake Up with Coffee  | 2028   | 21842.40              |
| Wake Up with Coffee  | 2029   | 9706.00               |


### Discussion

The core code for this analysis uses the groupby() method to aggregate the already calculated Item_Profit column, grouping the data first by Business_Name and then by Year. The resulting total profit for each period is then calculated using the .sum() function.

Observing the resulting table, the financial performance of the two businesses shows contrasting trends.

Ed's Barber Supplies demonstrated a pattern of increasing yearly profit between 2022 and 2024, reaching its highest point at 39,890.50 dollars. However, this positive trend was sharply reversed in 2025, where the business recorded its lowest profit of 11,739.90 dollars. This significant drop represents a decrease of more than 50% from the previous year, highlighting a potential operational or market disruption in 2025. Across the analysed period, Ed's Barber Supplies maintained a strong average annual profit of approximately 24,347.87 dollars.

Wake Up with Coffee, on the other hand, operates with a larger data set, exhibiting a broader range of financial results. Their maximum annual profit of 21,842.40 dollars was recorded in 2028, while their minimum profit of 6,514.70 dollars occurred earlier in 2022. Despite this wider fluctuation, the business maintained a steady average annual profit of approximately 16,995.24 dollars.

---

## Exploration Two

### Introduction

This analysis identifies the star performer: the single product that generated the highest total gross profit (Unit Profit × Quantity Sold) for each business in each year. This correctly accounts for volume, meaning a high-volume, lower-margin item can potentially beat a low-volume, high-margin item in total profit generation.

### Methods

In [24]:
print("\n Analysis for Focus Question 2: Most Profitable Product per Business per Year")

item_profit_summary = receipts_df.groupby(['Business_Name', 'Year', 'Item_Name'])['Item_Profit'].sum().reset_index()

idx_max_profit = item_profit_summary.groupby(['Business_Name', 'Year'])['Item_Profit'].idxmax()

most_profitable_item = item_profit_summary.loc[idx_max_profit].reset_index(drop=True)

most_profitable_item.rename(columns={'Item_Profit': 'Max_Item_Profit'}, inplace=True)
most_profitable_item = most_profitable_item[['Business_Name', 'Year', 'Item_Name', 'Max_Item_Profit']]

print(most_profitable_item.to_markdown(index=False, floatfmt=".2f", numalign="left", stralign="left"))


 Analysis for Focus Question 2: Most Profitable Product per Business per Year
| Business_Name        | Year   | Item_Name    | Max_Item_Profit   |
|:---------------------|:-------|:-------------|:------------------|
| Ed's Barber Supplies | 2022   | Cape grey    | 5010.70           |
| Ed's Barber Supplies | 2023   | Cape grey    | 11198.40          |
| Ed's Barber Supplies | 2024   | Cape grey    | 10551.60          |
| Ed's Barber Supplies | 2025   | Clippers kit | 3306.50           |
| Wake Up with Coffee  | 2022   | Eggs benny   | 2462.40           |
| Wake Up with Coffee  | 2023   | Ham bagel    | 2824.90           |
| Wake Up with Coffee  | 2024   | Salmon bagel | 4734.00           |
| Wake Up with Coffee  | 2025   | Salmon bagel | 5743.60           |
| Wake Up with Coffee  | 2026   | Cupcake      | 3375.10           |
| Wake Up with Coffee  | 2027   | Ham bagel    | 5015.20           |
| Wake Up with Coffee  | 2028   | Salmon bagel | 7484.40           |
| Wake Up with Coffee  |

### Discussion

This analysis utilises a two-step aggregation process:

First, it groups the data by Business_Name, Year, and Item_Name to sum the total profit for each distinct item.

Second, it uses the highly efficient idxmax() function on the result of the first grouping to select only the row corresponding to the highest profit for that specific business/year combination.

Ed's Barber Supplies shows a stable, high-profit performer that was abruptly replaced. Between the years 2022 and 2024, the business's most profitable item was the Cape grey product, with the profit from this single item exceeding 10,000 dollars in two of those years. This strong reliance on the Cape grey product was broken in 2025, when the Clippers kit became the most profitable product, reaching 3,306.50 dollars for that year. This change in the top-performing product coincides with the overall sharp decrease in annual profit observed in Question 1, suggesting that the profitability of the Cape grey product may have been affected.

For Wake Up with Coffee, the most profitable product changed frequently over the many years analysed, indicating a diversified profit stream rather than reliance on a single item. Products such as Cupcake, Salmon, and Ham bagel each made the most profitable list multiple times. On average, however, the Salmon bagel stands out as the highest earner, being the most profitable product with an average annual profit contribution of roughly 5,987.33 dollars.

---

## Exploration Three

### Introduction

Popularity is measured solely by volume, irrespective of profit or price. This metric reveals customer demand—the product most frequently purchased in terms of unit quantity for each business/year. Comparing the "Most Profitable" (Q2) and "Most Popular" (Q3) can highlight potential disconnects between customer preference and business revenue strategy.

### Methods

In [25]:
print("\n Analysis for Focus Question 3: Most Popular Product per Business per Year (by Quantity Sold)")
    
item_quantity_summary = receipts_df.groupby(['Business_Name', 'Year', 'Item_Name'])['Item_Quantity'].sum().reset_index()
    
idx_max_quantity = item_quantity_summary.groupby(['Business_Name', 'Year'])['Item_Quantity'].idxmax()
    
most_popular_item = item_quantity_summary.loc[idx_max_quantity].reset_index(drop=True)
    
most_popular_item.rename(columns={'Item_Quantity': 'Total_Units_Sold'}, inplace=True)
most_popular_item = most_popular_item[['Business_Name', 'Year', 'Item_Name', 'Total_Units_Sold']]
    

print(most_popular_item.to_markdown(index=False, numalign="left", stralign="left", 
        floatfmt={'Total_Units_Sold': '.0f'} 
    ))


 Analysis for Focus Question 3: Most Popular Product per Business per Year (by Quantity Sold)
| Business_Name        | Year   | Item_Name      | Total_Units_Sold   |
|:---------------------|:-------|:---------------|:-------------------|
| Ed's Barber Supplies | 2022   | Soap 500mL     | 175                |
| Ed's Barber Supplies | 2023   | Soap 500mL     | 363                |
| Ed's Barber Supplies | 2024   | Razorblade     | 324                |
| Ed's Barber Supplies | 2025   | Scissors       | 79                 |
| Wake Up with Coffee  | 2022   | Mocha          | 203                |
| Wake Up with Coffee  | 2023   | Ham toastie    | 411                |
| Wake Up with Coffee  | 2024   | Ham bagel      | 451                |
| Wake Up with Coffee  | 2025   | Salmon bagel   | 494                |
| Wake Up with Coffee  | 2026   | Long Black     | 564                |
| Wake Up with Coffee  | 2027   | Cupcake        | 570                |
| Wake Up with Coffee  | 2028   | Salmon 

### Discussion

This analysis is structurally identical to Question 2, but instead of summing Item_Profit, it sums Item_Quantity to measure sales volume. It then uses idxmax() on this quantity column (Total_Units_Sold) to find the product with the highest unit sales volume for the given period.

The popularity trends for Ed's Barber Supplies show a noticeable shift in customer purchasing behaviour over the analysed years. For 2022 and 2023, the most popular product was consistently the 500mL soap. This trend did not continue, as the most popular item changed annually thereafter: it was the Razorblades in 2024, and the Scissors in 2025. Across all three analyses (Total Profit, Most Profitable Item, and Most Popular Item), the final year (2025) for Ed's Barber Supplies shows a significantly worse performance compared to previous years. This consistent, sharp decline in key metrics strongly suggests a severe operational contraction, possibly indicating the business is undergoing a major restructuring, sell-off, or is preparing for closure.

Wake Up with Coffee exhibited high product turnover in terms of popularity. Unlike the profit analysis, where the Salmon Bagel recurred as the top earner, this business had a different top-selling product almost every year when measured by volume, with the Salmon Bagel being the only item to reappear multiple times as the most popular. 